In [1]:
import os
import sys
from tqdm import tqdm
import pickle
import numpy as np
from collections import defaultdict

import torch
from scipy.stats import zscore
from torchvision import transforms
sub_list = ['sub-01', 'sub-02', 'sub-04']
sessions = ['01', '02', '03', 'study', 'test', 'snap']

In [2]:
dic = {}

data_folder = '/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/'

folder_path = os.path.join(data_folder, 'afni')
for sub in sub_list:
    with open(f'{folder_path}/{sub}_roi_vox_all_sessions.pkl', 'rb') as file:
        dic[sub] = pickle.load(file)

In [3]:
# Masking data
for sub in sub_list:
    union_mask = dic[sub]['union_mask']
    print('NSD mask size:', union_mask.shape)
    print('union mask size:', sum(union_mask))
    for ses in sessions:
        dic[sub][ses]['roi'] = dic[sub][ses]['roi'][:, union_mask]
        # z-score each session
        dic[sub][ses]['roi'] = np.nan_to_num(zscore(dic[sub][ses]['roi'], axis=0))
        s = dic[sub][ses]['roi'].shape
        print(f'{ses}: {s}')

NSD mask size: (16946,)
union mask size: 5796
01: (693, 5796)
02: (693, 5796)
03: (693, 5796)
study: (216, 5796)
test: (216, 5796)
snap: (432, 5796)
NSD mask size: (17175,)
union mask size: 3643
01: (693, 3643)
02: (693, 3643)
03: (693, 3643)
study: (216, 3643)
test: (216, 3643)
snap: (432, 3643)
NSD mask size: (20397,)
union mask size: 5887
01: (693, 5887)
02: (693, 5887)
03: (693, 5887)
study: (216, 5887)
test: (216, 5887)
snap: (432, 5887)


In [4]:
# 455 * 3 + 26 (13 pairs; 3 repeats) + 80 (2 repeats per session_
unique_images = set(dic[sub]['01']['trial'] + dic[sub]['02']['trial'] + dic[sub]['03']['trial'])
test_images = [f'A_{i}' for i in range(1,19)] + [f'B_{i}' for i in range(1,19)]

In [5]:
import imageio.v2 as imageio
resize_transform = transforms.Resize((224, 224))

images = None

img_path = f'{folder_path}/loaded_mindeye_imgs.pkl'

if os.path.exists(img_path):
    with open(img_path, 'rb') as file:
        images = pickle.load(file)
        print('Loading image saved at: ', file)
else:
    for img in tqdm(unique_images):

        root_dir = os.path.join(data_folder, 'stimuli')
        if 'unchosen' in img:
            image_file = f'{root_dir}/unchosen_nsd_1000_images/{img}.png'
        elif 'special' in img and 'notspecial' not in img:
            image_file = f'{root_dir}/special515/{img}.jpg'
        elif 'notspecial' in img:
            image_file = f'{root_dir}/shared1000_notspecial/{img}.png'
        elif 'pair_' and '_w_' in img:
            image_file = f'{root_dir}/MST_pairs/{img}.jpg'
        else:
            print(img)

        if image_file and not os.path.exists(image_file):
            print('Cannot find the image at this path',image_file)
            break

        im = imageio.imread(image_file)
        im = torch.Tensor(im / 255).permute(2,0,1)
        im = resize_transform(im.unsqueeze(0))

        if images is None:
            images = im
        else:
            images = torch.vstack((images, im))

    
    print(folder_path)
    with open(img_path, 'wb') as file:
        pickle.dump(images, file)
        print('image saved at: ', file)
        
print("images", images.shape)

Loading image saved at:  <_io.BufferedReader name='/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/afni/loaded_mindeye_imgs.pkl'>
images torch.Size([1471, 3, 224, 224])


In [6]:
test_img = None

img_path = f'{folder_path}/loaded_test_imgs.pkl'

if os.path.exists(img_path):
    with open(img_path, 'rb') as file:
        test_img = pickle.load(file)
        print('Loading image saved at: ', file)
else:
    for img in test_images:

        root_dir = os.path.join(data_folder, 'stimuli', 'scenes')
        img_list = img.split('_')[0]
        img_id = int(img.split('_')[1])
        
        image_file = f'{root_dir}/list{img_list}/{img_id:02d}.png'

        if image_file and not os.path.exists(image_file):
            print('Cannot find the image at this path',image_file)
            break

        im = imageio.imread(image_file)
        im = torch.Tensor(im / 255).permute(2,0,1)
        im = resize_transform(im.unsqueeze(0))

        if test_img is None:
            test_img = im
        else:
            test_img = torch.vstack((test_img, im))

    
    print(folder_path)
    with open(img_path, 'wb') as file:
        pickle.dump(test_img, file)
        print('image saved at: ', file)
        
print("testing images", test_img.shape)

Loading image saved at:  <_io.BufferedReader name='/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/afni/loaded_test_imgs.pkl'>
testing images torch.Size([36, 3, 224, 224])


In [7]:
def find_repeated_strings(string_list):
    """
    Finds all repeated strings in a list and returns a dictionary 
    with the string as the key and a list of its indices as the value.
    Uses a set to track seen items efficiently.
    """
    # Set to quickly track which items have appeared once already
    seen_once = set()
    # Dictionary to store only the indices of items that repeat
    repeated_strings_dict = {}

    for index, string_val in enumerate(string_list):
        if string_val in repeated_strings_dict:
            # If already in the 'repeated_strings_dict', just append the new index
            repeated_strings_dict[string_val].append(index)
        elif string_val in seen_once:
            # First time seeing a repeat: move from 'seen_once' to 'repeated_strings_dict'
            repeated_strings_dict[string_val] = [string_list.index(string_val), index]
        else:
            # First time seeing the item overall
            seen_once.add(string_val)
            
    return repeated_strings_dict

def locate_repeat_index_per_run(sub_dict, unique_idx):
    
    vox = sub_dict['roi']
    runs = sub_dict['run']
    unique_runs = list(set(runs))
    trials = ["_".join(trial.split('_')[1:-1]) for trial in sub_dict['trial']]
    repeated_trial = find_repeated_strings(trials)

    # structure output idx dictionary
    unique_runs.sort()
    default_value = {}
    sorted_vox = dict.fromkeys(unique_runs, default_value)
    for k in sorted_vox.keys():
        sorted_vox[k] = defaultdict(list)
        
    for trial in test_images:
        idx_list = repeated_trial[trial]
        for i in idx_list:
            curr_run = runs[i]
            sorted_vox[curr_run][trial].append(i)
    
    return repeated_trial, sorted_vox

In [8]:
def average_repeats(vox, mindeye_trial, unique_images):
    
    repeated_trial = find_repeated_strings(mindeye_trial)
    
    sorted_vox = np.zeros((len(unique_images), vox.shape[1]))
    
    # Average repeated MST images
    for i, img in enumerate(unique_images):

        if img in repeated_trial.keys(): # deal with repeated images
            # average all repeats across sessions
            curr_trial_vox = np.mean(vox[repeated_trial[img]], axis=0)
            
        elif img in mindeye_trial: # deal with once images
            idx = mindeye_trial.index(img)
            curr_trial_vox = vox[idx, :]
            
        else: # error handeling
            print(f"{img} is not in the list")
            break
        
        sorted_vox[i, :] = curr_trial_vox
    
    return sorted_vox


def average_repeats_snap(vox, repeated_trial, unique_images):
        
    sorted_vox = np.zeros((len(unique_images), vox.shape[1]))
    assert len(repeated_trial.keys()) == len(unique_images)
    
    # Average repeated MST images
    for i, img in enumerate(unique_images):

        if img in repeated_trial.keys(): # deal with repeated images
            # average all repeats across sessions
            curr_trial_vox = np.mean(vox[repeated_trial[img]], axis=0)
            
        else: # error handeling
            print(f"{img} is not in the list")
            break
        
        sorted_vox[i, :] = curr_trial_vox
    
    return sorted_vox

In [9]:
# Stacking multi-session data:
vox_data = {}

for sub in sub_list:
    mindeye_vox = np.vstack((dic[sub]['01']['roi'],dic[sub]['02']['roi'],dic[sub]['03']['roi']))
    mindeye_trial = dic[sub]['01']['trial']+dic[sub]['02']['trial']+dic[sub]['03']['trial']
    mindeye_vox = average_repeats(mindeye_vox, mindeye_trial, unique_images)
    vox_data[sub] = mindeye_vox
    
    break

In [10]:
# Loading Snap data
tasks = ['study', 'test', 'snap']

test_data = {}

for sub in sub_list:
    test_data[sub] = {}
    
    for task in tasks:
        
        sub_dict = dic[sub][task]
        
        repeat_idx, per_run_repeat_idx = locate_repeat_index_per_run(sub_dict, test_images)

        test_data[sub][task] = average_repeats_snap(sub_dict['roi'], repeat_idx, test_images)

    break

### Testing single subject

In [11]:
train_images = torch.Tensor(images)
train_vox = torch.Tensor(vox_data['sub-01'])
assert len(train_images) == len(train_vox)

In [12]:
test_images = torch.Tensor(test_img)
test_vox = torch.Tensor(np.mean([test_data['sub-01']['study'], test_data['sub-01']['test'], test_data['sub-01']['snap']], axis=0))
test_vox_study = torch.Tensor(test_data['sub-01']['study'])
test_vox_test = torch.Tensor(test_data['sub-01']['test'])
test_vox_snap = torch.Tensor(test_data['sub-01']['snap'])
assert len(test_images) == len(test_vox)

In [13]:
assert train_vox.shape[1] == test_vox.shape[1]

## Finished loading data. Setting up GPU

In [14]:
### Multi-GPU config ###
from accelerate import Accelerator, DeepSpeedPlugin

local_rank = os.getenv('RANK')
if local_rank is None: 
    local_rank = 0
else:
    local_rank = int(local_rank)
print("LOCAL RANK ", local_rank)  

data_type = torch.float32 # change depending on your mixed_precision

accelerator = Accelerator(split_batches=False)
batch_size = 8

LOCAL RANK  0


In [15]:
print("PID of this process =",os.getpid())
device = accelerator.device
print("device:",device)
world_size = accelerator.state.num_processes
distributed = not accelerator.state.distributed_type == 'NO'
num_devices = torch.cuda.device_count()
global_batch_size = batch_size * num_devices
print("global_batch_size", global_batch_size)
if num_devices==0 or not distributed: num_devices = 1
num_workers = num_devices
print(accelerator.state)

# set data_type to match your mixed precision (automatically set based on deepspeed config)
if accelerator.mixed_precision == "bf16":
    data_type = torch.bfloat16
elif accelerator.mixed_precision == "fp16":
    data_type = torch.float16
else:
    data_type = torch.float32

print("distributed =",distributed, "num_devices =", num_devices, "local rank =", local_rank, "world size =", world_size, "data_type =", data_type)
print = accelerator.print # only print if local_rank=0

PID of this process = 1120570
device: cuda
global_batch_size 8
Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: no

distributed = False num_devices = 1 local rank = 0 world size = 1 data_type = torch.float32


In [16]:
## USING OpenCLIP ViT-bigG ###
sys.path.append('generative_models/')
import sgm
from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder
# from generative_models.sgm.models.diffusion import DiffusionEngine
# from omegaconf import OmegaConf

In [17]:
try:
    print(clip_img_embedder)
except:
    clip_img_embedder = FrozenOpenCLIPImageEmbedder(
        arch="ViT-bigG-14",
        version="laion2b_s39b_b160k",
        output_tokens=True,
        only_tokens=True,
    )
    clip_img_embedder.to(device)
clip_img_embedder.model.visual.set_grad_checkpointing(True)
clip_seq_dim = 256
clip_emb_dim = 1664

In [18]:
num_voxels_list=[train_vox[0].shape[-1]]
n_blocks=4
hidden_dim=1024
use_prior=False
clip_scale=1.

In [19]:
import utils
from models import PriorNetwork, BrainDiffusionPrior

In [20]:
model = utils.prepare_model_and_training(
    num_voxels_list=num_voxels_list,
    n_blocks=n_blocks,
    hidden_dim=hidden_dim,
    clip_emb_dim=clip_emb_dim,
    clip_seq_dim=clip_seq_dim,
    use_prior=use_prior,
    clip_scale=clip_scale
)

MindEyeModule()
param counts:
5,936,128 total
5,936,128 trainable
param counts:
5,936,128 total
5,936,128 trainable
param counts:
453,360,280 total
453,360,280 trainable
param counts:
459,296,408 total
459,296,408 trainable


In [21]:
# test on subject 1 with fake data
b = torch.randn((2,1,num_voxels_list[0]))
print(b.shape, model.ridge(b,0).shape)

torch.Size([2, 1, 5796]) torch.Size([2, 1, 1024])


In [22]:
# test that the model works on some fake data
b = torch.randn((2,1,hidden_dim))
print("b.shape",b.shape)

backbone_, clip_, blur_ = model.backbone(b)
print(backbone_.shape, clip_.shape, blur_[0].shape, blur_[1].shape)

b.shape torch.Size([2, 1, 1024])
torch.Size([2, 256, 1664]) torch.Size([2, 256, 1664]) torch.Size([1]) torch.Size([1])


## Setup optimizer / lr / ckpt saving

In [58]:
prior_lr=3e-4
lr_scheduler_type='cycle'
max_lr=3e-4
num_iterations_per_epoch=len(train_images)//batch_size
num_epochs=30

ckpt_saving=True
import time
ts = time.time()
outdir = os.path.join(data_folder, f'output_{ts}')
if not os.path.exists(outdir) and ckpt_saving:
    os.makedirs(outdir,exist_ok=True)

In [59]:
num_iterations_per_epoch

183

In [60]:
no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']

opt_grouped_parameters = [
    {'params': [p for n, p in model.ridge.named_parameters()], 'weight_decay': 1e-2},
    {'params': [p for n, p in model.backbone.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 1e-2},
    {'params': [p for n, p in model.backbone.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
# model.backbone.requires_grad_(False)

if use_prior:
    effective_prior_lr = prior_lr if prior_lr is not None else max_lr
    print(f"--- Setting learning rate for diffusion_prior: {effective_prior_lr} ---")

    if prior_lr is not None:
        assert lr_scheduler_type == 'cycle'  # if prior_lr exists, ensure lr scheduler is cycle because we want to set custom lr for the prior. custom lr for prior is not implemented in the linear scheduler code.

    opt_grouped_parameters.extend([
        {'params': [p for n, p in model.diffusion_prior.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 1e-2, 'lr': effective_prior_lr},
        {'params': [p for n, p in model.diffusion_prior.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0, 'lr': effective_prior_lr}
    ])

optimizer = torch.optim.AdamW(opt_grouped_parameters, lr=max_lr)

if lr_scheduler_type == 'linear':
    lr_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        total_iters=int(np.floor(num_epochs*num_iterations_per_epoch)),
        last_epoch=-1
    )
elif lr_scheduler_type == 'cycle':
    if num_iterations_per_epoch==0:
        num_iterations_per_epoch=1
    total_steps=int(np.floor(num_epochs*num_iterations_per_epoch))
    print("total_steps", total_steps)
    max_lrs = [max_lr] * 3  # for ridge and backbone
    if use_prior:
        max_lrs.extend([effective_prior_lr] * 2) # for prior

    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=max_lrs,
        total_steps=total_steps,
        final_div_factor=1000,
        last_epoch=-1, pct_start=2/num_epochs
    )
    
def save_ckpt(tag):
    ckpt_path = outdir+f'/{tag}.pth'
    if accelerator.is_main_process:
        unwrapped_model = accelerator.unwrap_model(model)
        torch.save({
            'epoch': epoch,
            'model_state_dict': unwrapped_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler': lr_scheduler.state_dict(),
            'train_losses': losses,
            'test_losses': test_losses,
            'lrs': lrs,
            }, ckpt_path)
    print(f"\n---saved {outdir}/{tag} ckpt!---\n")

def load_ckpt(tag,load_lr=True,load_optimizer=True,load_epoch=True,strict=True,outdir=outdir,multisubj_loading=False): 
    print(f"\n---loading {outdir}/{tag}.pth ckpt---\n")
    checkpoint = torch.load(outdir+'/last.pth', map_location='cpu')
    state_dict = checkpoint['model_state_dict']
    if multisubj_loading: # remove incompatible ridge layer that will otherwise error
        state_dict.pop('ridge.linears.0.weight',None)
    model.load_state_dict(state_dict, strict=strict)
    if load_epoch:
        globals()["epoch"] = checkpoint['epoch']
        print("Epoch",epoch)
    if load_optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if load_lr:
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
    del checkpoint

print("\nDone with model preparations!")
num_params = utils.count_params(model)

total_steps 5490

Done with model preparations!
param counts:
459,296,408 total
459,296,408 trainable


In [61]:
epoch = 0
losses, test_losses, lrs = [], [], []
best_test_loss = 1e9
torch.cuda.empty_cache()
# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

train_data = torch.utils.data.TensorDataset(torch.tensor(range(len(train_vox))))
train_dl = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=True)

In [62]:
test_data = torch.utils.data.TensorDataset(torch.tensor(range(len(test_vox))))
test_dl = torch.utils.data.DataLoader(test_data, batch_size=36, shuffle=False, drop_last=True, pin_memory=True)

In [63]:
model, optimizer, train_dl, lr_scheduler = accelerator.prepare(model, optimizer, train_dl, lr_scheduler)

In [64]:
model_name = 'training_mindeye' 
mixup_pct=.33
use_image_aug=False
import torch.nn as nn

In [65]:
for train_i, behav in enumerate(train_dl):  
    print(train_i)
    print(behav)
    print(behav[0])


0
[tensor([ 439,  589,  556, 1110,  871, 1154,  393, 1064], device='cuda:0')]
tensor([ 439,  589,  556, 1110,  871, 1154,  393, 1064], device='cuda:0')
1
[tensor([1285,  794, 1131,  833, 1410,  238,  395,  716], device='cuda:0')]
tensor([1285,  794, 1131,  833, 1410,  238,  395,  716], device='cuda:0')
2
[tensor([1395,  299,  484,  243, 1041,  946,  446,   78], device='cuda:0')]
tensor([1395,  299,  484,  243, 1041,  946,  446,   78], device='cuda:0')
3
[tensor([ 144,  404, 1194,   23, 1032,  588,  836,   73], device='cuda:0')]
tensor([ 144,  404, 1194,   23, 1032,  588,  836,   73], device='cuda:0')
4
[tensor([ 280,  244, 1378,  842, 1239,  580, 1152,   13], device='cuda:0')]
tensor([ 280,  244, 1378,  842, 1239,  580, 1152,   13], device='cuda:0')
5
[tensor([ 265,  658,  240, 1339, 1202, 1396,  975,   47], device='cuda:0')]
tensor([ 265,  658,  240, 1339, 1202, 1396,  975,   47], device='cuda:0')
6
[tensor([ 994,  835,  128, 1461, 1452, 1255, 1253,  188], device='cuda:0')]
tensor([ 9

In [66]:
    
for test_i, behav in enumerate(test_dl):  
    print(test_i)
    print(behav)
    break

0
[tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])]


In [67]:
blurry_recon = False
ckpt_interval = 99
wandb_log = False

In [68]:
epoch

0

In [69]:
print(f"{model_name} starting with epoch {epoch} / {num_epochs}")
progress_bar = tqdm(range(epoch,num_epochs), ncols=1200, disable=(local_rank!=0))
test_image, test_voxel = None, None
mse = nn.MSELoss()
l1 = nn.L1Loss()
soft_loss_temps = utils.cosine_anneal(0.004, 0.0075, num_epochs - int(mixup_pct * num_epochs))
skip_train = True if epoch>=(num_epochs-1) else False # skip training if you are resuming from a fully trained model

images = train_images
vox = train_vox

for epoch in progress_bar:
    model.train()

    fwd_percent_correct = 0.
    bwd_percent_correct = 0.
    test_fwd_percent_correct = 0.
    test_bwd_percent_correct = 0.
    
    recon_cossim = 0.
    test_recon_cossim = 0.
    recon_mse = 0.
    test_recon_mse = 0.

    loss_clip_total = 0.
    loss_blurry_total = 0.
    loss_blurry_cont_total = 0.
    test_loss_clip_total = 0.
    
    loss_prior_total = 0.
    test_loss_prior_total = 0.

    blurry_pixcorr = 0.
    test_blurry_pixcorr = 0. 

    # you now have voxel_iters and image_iters with num_iterations_per_epoch batches each
    for train_i, behav in enumerate(train_dl):  
        with torch.cuda.amp.autocast(dtype=data_type):
            optimizer.zero_grad()
            loss = 0.
            
            behav = behav[0]

            image = images[behav.long().cpu()].to(device)
            voxel = vox[behav.long().cpu()]

            # voxel = (voxel - train_mean) / train_std
            voxel = torch.Tensor(voxel).unsqueeze(1).to(device)

            if use_image_aug: 
                image = img_augment(image)

            clip_target = clip_img_embedder(image)
            assert not torch.any(torch.isnan(clip_target))

            if epoch < int(mixup_pct * num_epochs):
                voxel, perm, betas, select = utils.mixco(voxel)

            voxel_ridge = model.ridge(voxel,0) #[model.ridge(voxel_list[si],si) for si,s in enumerate(subj_list)]
            # voxel_ridge = torch.cat(voxel_ridge_list, dim=0)

            backbone, clip_voxels, blurry_image_enc_ = model.backbone(voxel_ridge)

            if clip_scale>0:
                clip_voxels_norm = nn.functional.normalize(clip_voxels.flatten(1), dim=-1)
                clip_target_norm = nn.functional.normalize(clip_target.flatten(1), dim=-1)

            if use_prior:
                loss_prior, prior_out = model.diffusion_prior(text_embed=backbone, image_embed=clip_target)
                loss_prior_total += loss_prior.item()
                loss_prior *= prior_scale
                loss += loss_prior

                recon_cossim += nn.functional.cosine_similarity(prior_out, clip_target).mean().item()
                recon_mse += mse(prior_out, clip_target).item()

            if clip_scale>0:
                if epoch < int(mixup_pct * num_epochs):                
                    loss_clip = utils.mixco_nce(
                        clip_voxels_norm,
                        clip_target_norm,
                        temp=.006,
                        perm=perm, betas=betas, select=select)
                else:
                    epoch_temp = soft_loss_temps[epoch-int(mixup_pct*num_epochs)]
                    loss_clip = utils.soft_clip_loss(
                        clip_voxels_norm,
                        clip_target_norm,
                        temp=epoch_temp)

                loss_clip_total += loss_clip.item()
                loss_clip *= clip_scale
                loss += loss_clip

            if blurry_recon:     
                image_enc_pred, transformer_feats = blurry_image_enc_

                image_enc = autoenc.encode(2*image-1).latent_dist.mode() * 0.18215
                loss_blurry = l1(image_enc_pred, image_enc)
                loss_blurry_total += loss_blurry.item()

                if epoch < int(mixup_pct * num_epochs):
                    image_enc_shuf = image_enc[perm]
                    betas_shape = [-1] + [1]*(len(image_enc.shape)-1)
                    image_enc[select] = image_enc[select] * betas[select].reshape(*betas_shape) + \
                        image_enc_shuf[select] * (1 - betas[select]).reshape(*betas_shape)

                image_norm = (image - mean)/std
                image_aug = (blur_augs(image) - mean)/std
                _, cnx_embeds = cnx(image_norm)
                _, cnx_aug_embeds = cnx(image_aug)

                cont_loss = utils.soft_cont_loss(
                    nn.functional.normalize(transformer_feats.reshape(-1, transformer_feats.shape[-1]), dim=-1),
                    nn.functional.normalize(cnx_embeds.reshape(-1, cnx_embeds.shape[-1]), dim=-1),
                    nn.functional.normalize(cnx_aug_embeds.reshape(-1, cnx_embeds.shape[-1]), dim=-1),
                    temp=0.2)
                loss_blurry_cont_total += cont_loss.item()

                loss += (loss_blurry + 0.1*cont_loss) * blur_scale #/.18215

            if clip_scale>0:
                # forward and backward top 1 accuracy        
                labels = torch.arange(len(clip_voxels_norm)).to(clip_voxels_norm.device) 
                fwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_voxels_norm, clip_target_norm), labels, k=1).item()
                bwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_target_norm, clip_voxels_norm), labels, k=1).item()

            if blurry_recon:
                with torch.no_grad():
                    # only doing pixcorr eval on a subset of the samples per batch because its costly & slow to compute autoenc.decode()
                    random_samps = np.random.choice(np.arange(len(image)), size=len(image)//5, replace=False)
                    blurry_recon_images = (autoenc.decode(image_enc_pred[random_samps]/0.18215).sample/ 2 + 0.5).clamp(0,1)
                    pixcorr = utils.pixcorr(image[random_samps], blurry_recon_images)
                    blurry_pixcorr += pixcorr.item()
            
            utils.check_loss(loss)
            accelerator.backward(loss)
            optimizer.step()

            losses.append(loss.item())
            lrs.append(optimizer.param_groups[0]['lr'])

            if lr_scheduler_type is not None:
                lr_scheduler.step()
                
            if train_i >= num_iterations_per_epoch-1:
                break
                
    model.eval()
    if local_rank==0:
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=data_type): 
            for test_i, behav in enumerate(test_dl):  
                behav = behav[0]

                loss=0.

                if behav.ndim>1:
                    image = images[behav[:,0].long().cpu()].to(device)
                    voxel = vox[behav.long().cpu()].mean(1)
                else:
                    image = images[behav.long().cpu()].to(device)
                    voxel = vox[behav.long().cpu()]
                    
                voxel = torch.Tensor(voxel).unsqueeze(1).to(device)

                clip_img_embedder = clip_img_embedder.to(device)
                clip_target = clip_img_embedder(image.float())
                
                voxel_ridge = model.ridge(voxel,0)

                backbone, clip_voxels, blurry_image_enc_ = model.backbone(voxel_ridge)

                if clip_scale>0:
                    clip_voxels_norm = nn.functional.normalize(clip_voxels.flatten(1), dim=-1)
                    clip_target_norm = nn.functional.normalize(clip_target.flatten(1), dim=-1)
                
                # for some evals, only doing a subset of the samples per batch because of computational cost
                random_samps = np.random.choice(np.arange(len(image)), size=len(image)//5, replace=False)
                
                if use_prior:
                    loss_prior, contaminated_prior_out = model.diffusion_prior(text_embed=backbone[random_samps], image_embed=clip_target[random_samps])
                    test_loss_prior_total += loss_prior.item()
                    loss_prior *= prior_scale
                    loss += loss_prior
                        
                if clip_scale>0:
                    loss_clip = utils.soft_clip_loss(
                        clip_voxels_norm,
                        clip_target_norm,
                        temp=.006)

                    test_loss_clip_total += loss_clip.item()
                    loss_clip = loss_clip * clip_scale
                    loss += loss_clip

                if blurry_recon:
                    image_enc_pred, _ = blurry_image_enc_
                    blurry_recon_images = (autoenc.decode(image_enc_pred[random_samps]/0.18215).sample / 2 + 0.5).clamp(0,1)
                    pixcorr = utils.pixcorr(image[random_samps], blurry_recon_images)
                    test_blurry_pixcorr += pixcorr.item()

                if clip_scale>0:
                    # forward and backward top 1 accuracy        
                    labels = torch.arange(len(clip_voxels_norm)).to(clip_voxels_norm.device) 
                    test_fwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_voxels_norm, clip_target_norm), labels, k=1).item()
                    test_bwd_percent_correct += utils.topk(utils.batchwise_cosine_similarity(clip_target_norm, clip_voxels_norm), labels, k=1).item()
                
                utils.check_loss(loss)                
                test_losses.append(loss.item())

            # if utils.is_interactive(): clear_output(wait=True)
            if skip_train: break
            print("---")

            # assert (test_i+1) == 1
            logs = {"train/loss": np.mean(losses[-(train_i+1):]),
                "test/loss": np.mean(test_losses[-(test_i+1):]),
                "train/lr": lrs[-1],
                "train/num_steps": len(losses),
                "test/num_steps": len(test_losses),
                "train/fwd_pct_correct": fwd_percent_correct / (train_i + 1),
                "train/bwd_pct_correct": bwd_percent_correct / (train_i + 1),
                "test/test_fwd_pct_correct": test_fwd_percent_correct / (test_i + 1),
                "test/test_bwd_pct_correct": test_bwd_percent_correct / (test_i + 1),
                "train/loss_clip_total": loss_clip_total / (train_i + 1),
                "train/loss_blurry_total": loss_blurry_total / (train_i + 1),
                "train/loss_blurry_cont_total": loss_blurry_cont_total / (train_i + 1),
                "test/loss_clip_total": test_loss_clip_total / (test_i + 1),
                "train/blurry_pixcorr": blurry_pixcorr / (train_i + 1),
                "test/blurry_pixcorr": test_blurry_pixcorr / (test_i + 1),
                "train/recon_cossim": recon_cossim / (train_i + 1),
                "test/recon_cossim": test_recon_cossim / (test_i + 1),
                "train/recon_mse": recon_mse / (train_i + 1),
                "test/recon_mse": test_recon_mse / (test_i + 1),
                "train/loss_prior": loss_prior_total / (train_i + 1),
                "test/loss_prior": test_loss_prior_total / (test_i + 1),
                }

            # if finished training, save jpg recons if they exist
            if (epoch == num_epochs-1) or (epoch % ckpt_interval == 0):
                if blurry_recon:    
                    image_enc = autoenc.encode(2*image[:4]-1).latent_dist.mode() * 0.18215
                    # transform blurry recon latents to images and plot it
                    fig, axes = plt.subplots(1, 8, figsize=(10, 4))
                    jj=-1
                    for j in [0,1,2,3]:
                        jj+=1
                        axes[jj].imshow(utils.torch_to_Image((autoenc.decode(image_enc[[j]]/0.18215).sample / 2 + 0.5).clamp(0,1)))
                        axes[jj].axis('off')
                        jj+=1
                        axes[jj].imshow(utils.torch_to_Image((autoenc.decode(image_enc_pred[[j]]/0.18215).sample / 2 + 0.5).clamp(0,1)))
                        axes[jj].axis('off')
                    plt.show()

            progress_bar.set_postfix(**logs)

            if wandb_log: wandb.log(logs)
            
    # Save model checkpoint and reconstruct
    if (ckpt_saving) and (epoch % ckpt_interval == 0):
        save_ckpt(f'last')

    # wait for other GPUs to catch up if needed
    accelerator.wait_for_everyone()
    torch.cuda.empty_cache()

print("\n===Finished!===\n")
if ckpt_saving:
    save_ckpt(f'last')

training_mindeye starting with epoch 0 / 30


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

---


  3%|██████████████████████▎                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | 1/30 [00:33<16:00, 33.11s/it, test/blurry_pixcorr=0, test/loss=3.8, test/loss_clip_total=3.8, test/loss_prior=0, test/num_steps=1, test/recon_cossim=0, test/recon_mse=0, test/test_bwd_pct_correct=0.0556, test/test_fwd_pct_correct=0.278, train/blurry_pixcorr=0, train/bwd_pct_correct=0.127, train/fwd_pct_correct=0.145, trai


---saved /scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/output_1771364936.705772/last ckpt!---



...

 97%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                      | 29/30 [14:19<00:29, 29.43s/it, test/blurry_pixcorr=0, test/loss=0.0129, test/loss_clip_total=0.0129, test/loss_prior=0, test/num_steps=29, test/recon_cossim=0, test/recon_mse=0, test/test_bwd_pct_correct=1, test/test_fwd_pct_correct=1, train/blurry_pixcorr=0, train/bwd_pct_correct=1, train/fwd_pct_correct=1, train/loss

---


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [14:48<00:00, 29.62s/it, test/blurry_pixcorr=0, test/loss=0.0129, test/loss_clip_total=0.0129, test/loss_prior=0, test/num_steps=30, test/recon_cossim=0, test/recon_mse=0, test/test_bwd_pct_correct=1, test/test_fwd_pct_correct=1, train/blurry_pixcorr=0, train/bwd_pct_correct=1, train/fwd_pct_correct=1, train/los

---

===Finished!===




---saved /scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/output_1771364936.705772/last ckpt!---

